# 02 — Distribution Fitting

Before we can run a Monte Carlo CLV simulation, we need a model of customer behaviour
that we can *sample from* — not just a table of past observations.  
This notebook is that bridge: we take the raw synthetic data and fit statistical
distributions to it, then verify the fits with formal goodness-of-fit tests.

The two distributions we care about are:

| Variable | Distribution | Why |
|---|---|---|
| `monthly_spending` | **Gamma** | Right-skewed, positive support, two interpretable parameters |
| `months_to_churn`  | **Weibull** | Models time-to-event with a shape-dependent hazard rate |

By the end of this notebook we will have saved a `reports/fitted_params.json` file
that the simulation notebook (`03_monte_carlo_clv.ipynb`) will load directly.

---
## 0 — Imports

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import gamma as gamma_fn

# Consistent colours per segment throughout the notebook
SEGMENT_COLOURS = {
    'high_value': '#2196F3',   # blue
    'standard':   '#4CAF50',   # green
    'churn_risk': '#F44336',   # red
}

SEGMENT_ORDER = ['high_value', 'standard', 'churn_risk']

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

---
## 1 — Load Data

We read the same CSV produced by `src/data_generator.py`.  
The columns of interest here are `segment`, `monthly_spending`, and `months_to_churn`.

We also capture the **true parameters** that the generator used, so we can compare
what we *know* was used to create the data against what the fitter *discovers* from
the sample alone.  This comparison is the clearest demonstration of what fitting means:
given only the observations, we should recover the generating parameters closely.

In [ ]:
DATA_PATH   = pathlib.Path('../reports/synthetic_customers.csv')
REPORTS_DIR = pathlib.Path('../reports')
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(f'Loaded {len(df):,} customers across {df["segment"].nunique()} segments')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# True parameters used by data_generator.py—our fitting should recover these closely.
TRUE_PARAMS = {
    'high_value': {'gamma': {'alpha': 50, 'beta': 10}, 'weibull': {'shape': 1.2, 'scale': 40}},
    'standard':   {'gamma': {'alpha': 30, 'beta':  5}, 'weibull': {'shape': 1.0, 'scale': 25}},
    'churn_risk': {'gamma': {'alpha': 15, 'beta':  3}, 'weibull': {'shape': 0.8, 'scale': 10}},
}

print('True parameters (for post-fit comparison):')
for seg, p in TRUE_PARAMS.items():
    g, w = p['gamma'], p['weibull']
    print(f'  {seg:12s}  Gamma(alpha={g["alpha"]}, beta={g["beta"]})'  
          f'  Weibull(shape={w["shape"]}, scale={w["scale"]})')

---
## 2 — Gamma Distribution Fitting: `monthly_spending`

### What does fitting a distribution mean?

A distribution is a mathematical formula that describes the probability of observing
any particular value.  It is fully defined by its **parameters** — a small set of
numbers that control the shape and location.

**Fitting** a distribution means: *given a set of observed data points, find the
parameter values that make those observations most probable under that distribution.*
The standard method is **Maximum Likelihood Estimation (MLE)**, which is what
`scipy.stats.*.fit()` does under the hood.

Once fitted, the distribution is no longer just a description of the past — it is a
generative model we can **sample from** to create hypothetical future customers in the
Monte Carlo simulation.

---

### What do alpha and beta control in the Gamma distribution?

| Parameter | Symbol | Controls |
|---|---|---|
| **Alpha** (shape) | α | How peaked vs. spread-out the distribution is. High α → bell-shaped, low variance. Low α → heavily right-skewed. |
| **Beta** (scale)  | β | The *scale* of the values. Multiplying β by 2 doubles all the values proportionally. |

Key formulas:
- **Mean** = α × β  
- **Variance** = α × β²  
- **Coefficient of variation** = 1 / √α

So Gamma(α=50, β=10) gives mean = R 500 and CV ≈ 0.14 (low spread), while
Gamma(α=15, β=3) gives mean = R 45 and CV ≈ 0.26 (more spread relative to the mean).

**`scipy` convention:** `scipy.stats.gamma.fit(data, floc=0)` returns `(a, loc, scale)`
where `a` = α and `scale` = β.  We fix `loc=0` because spending cannot be negative.

In [ ]:
gamma_params = {}  # fitted params for all segments

header = f"{'Segment':<14} {'Fit alpha':>10} {'Fit beta':>10} {'True alpha':>11} {'True beta':>10} {'Fit mean':>10} {'Data mean':>10}"
print(header)
print('-' * len(header))

for seg in SEGMENT_ORDER:
    data = df.loc[df['segment'] == seg, 'monthly_spending'].values

    # floc=0 constrains the location parameter to zero (spending >= 0)
    alpha_fit, _loc, beta_fit = stats.gamma.fit(data, floc=0)

    gamma_params[seg] = {'alpha': round(alpha_fit, 4), 'beta': round(beta_fit, 4)}

    true_a = TRUE_PARAMS[seg]['gamma']['alpha']
    true_b = TRUE_PARAMS[seg]['gamma']['beta']

    print(f'{seg:<14} {alpha_fit:>10.3f} {beta_fit:>10.3f} {true_a:>11} {true_b:>10}'
          f' R{alpha_fit * beta_fit:>9.2f} R{data.mean():>9.2f}')

print()
print('Fitted alpha and beta should be close to the True values.')
print('Small differences are expected—we are estimating from a finite sample.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, seg in zip(axes, SEGMENT_ORDER):
    data   = df.loc[df['segment'] == seg, 'monthly_spending'].values
    colour = SEGMENT_COLOURS[seg]
    alpha_fit = gamma_params[seg]['alpha']
    beta_fit  = gamma_params[seg]['beta']

    # Histogram with density=True so the y-axis matches the PDF
    ax.hist(data, bins=50, density=True, color=colour, alpha=0.45,
            edgecolor='none', label='observed data')

    # Fitted PDF curve
    x   = np.linspace(data.min(), data.max(), 400)
    pdf = stats.gamma.pdf(x, a=alpha_fit, loc=0, scale=beta_fit)
    ax.plot(x, pdf, color=colour, linewidth=2.5,
            label=f'Gamma(\u03b1={alpha_fit:.1f}, \u03b2={beta_fit:.1f})')

    ax.set_title(seg.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Monthly Spending (R)', fontsize=10)
    ax.set_ylabel('Probability Density', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R{v:,.0f}'))
    ax.legend(fontsize=8)

fig.suptitle('Gamma Distribution Fits \u2014 Monthly Spending by Segment',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gamma_fits.png', bbox_inches='tight')
plt.show()

---
## 3 — Weibull Distribution Fitting: `months_to_churn`

### Why Weibull for churn timing?

The Weibull distribution is the standard model for *time-to-event* data in reliability
engineering and survival analysis.  Its power comes from the **shape parameter k**,
which controls how the risk of the event (churning) changes over time:

| Shape (k) | Hazard rate | Real-world meaning |
|---|---|---|
| k < 1 | Decreasing | Infant mortality — highest churn risk right after acquisition |
| k = 1 | Constant | Memoryless (exponential) — churn risk does not depend on tenure |
| k > 1 | Increasing | Wear-out — customers become progressively more likely to churn |

### What do shape and scale control?

| Parameter | Controls |
|---|---|
| **Shape** (k) | The *character* of churn — whether the hazard rate is increasing, constant, or decreasing over time |
| **Scale** (λ) | The characteristic lifetime — larger scale shifts the distribution to the right (customers survive longer on average) |

Key formulas:
- **Mean** = λ · Γ(1 + 1/k)  (where Γ is the Gamma function)
- **Median** = λ · (ln 2)^(1/k)

**`scipy` convention:** `scipy.stats.weibull_min.fit(data, floc=0)` returns `(c, loc, scale)`
where `c` = shape (k) and `scale` = λ.  We fix `loc=0` because time cannot be negative.

In [ ]:
weibull_params = {}  # fitted params for all segments

header = f"{'Segment':<14} {'Fit k':>8} {'Fit lambda':>11} {'True k':>7} {'True lambda':>12} {'Fit mean':>10} {'Data mean':>10}"
print(header)
print('-' * len(header))

for seg in SEGMENT_ORDER:
    data = df.loc[df['segment'] == seg, 'months_to_churn'].values

    # floc=0 constrains the location to zero (time to churn >= 0)
    shape_fit, _loc, scale_fit = stats.weibull_min.fit(data, floc=0)

    weibull_params[seg] = {'shape': round(shape_fit, 4), 'scale': round(scale_fit, 4)}

    true_k = TRUE_PARAMS[seg]['weibull']['shape']
    true_s = TRUE_PARAMS[seg]['weibull']['scale']

    # Weibull mean: scale * Gamma(1 + 1/shape)
    fit_mean = scale_fit * gamma_fn(1.0 + 1.0 / shape_fit)

    print(f'{seg:<14} {shape_fit:>8.3f} {scale_fit:>11.3f} {true_k:>7} {true_s:>12}'
          f' {fit_mean:>9.2f}mo {data.mean():>9.2f}mo')

print()
print('k < 1: decreasing hazard (churn_risk — infant mortality)')
print('k = 1: constant hazard / memoryless (standard)')
print('k > 1: increasing hazard / wear-out (high_value)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, seg in zip(axes, SEGMENT_ORDER):
    data      = df.loc[df['segment'] == seg, 'months_to_churn'].values
    colour    = SEGMENT_COLOURS[seg]
    shape_fit = weibull_params[seg]['shape']
    scale_fit = weibull_params[seg]['scale']

    # Histogram with density=True
    ax.hist(data, bins=50, density=True, color=colour, alpha=0.45,
            edgecolor='none', label='observed data')

    # Fitted PDF curve
    x   = np.linspace(0.01, data.max(), 400)
    pdf = stats.weibull_min.pdf(x, c=shape_fit, loc=0, scale=scale_fit)
    ax.plot(x, pdf, color=colour, linewidth=2.5,
            label=f'Weibull(k={shape_fit:.2f}, \u03bb={scale_fit:.1f})')

    ax.set_title(seg.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Months to Churn', fontsize=10)
    ax.set_ylabel('Probability Density', fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('Weibull Distribution Fits \u2014 Months to Churn by Segment',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'weibull_fits.png', bbox_inches='tight')
plt.show()

---
## 4 — Goodness-of-Fit: Kolmogorov-Smirnov Test

### What is the KS test?

Visually, our fitted curves look close to the histograms — but looks are not a rigorous
criterion.  The **Kolmogorov-Smirnov (KS) test** quantifies this.

It works by comparing two cumulative distribution functions (CDFs):
- The **empirical CDF** — built directly from the data by sorting values and stepping up
  by 1/n at each observation.
- The **fitted CDF** — the theoretical CDF evaluated with the parameters we just found.

The **KS statistic D** is the *maximum vertical gap* between these two CDFs:

$$D = \sup_x \left| \hat{F}_n(x) - F(x; \hat{\theta}) \right|$$

A small D means the fit is tight everywhere; a large D points to a region where the
theoretical CDF diverges from the data.

### How to interpret the KS result

The **p-value** answers: *If the data truly came from this fitted distribution, how
probable is a KS statistic at least this extreme?*

| Result | Interpretation |
|---|---|
| p > 0.05 | Fail to reject H₀ — the data is consistent with the fitted distribution |
| p ≤ 0.05 | Reject H₀ — statistically significant evidence against the fitted distribution |

> **Important caveat:** With large samples (n > 1 000), the KS test becomes very
> sensitive.  Even tiny, practically irrelevant deviations (e.g. rounding in the CSV)
> can produce p < 0.05.  A passing result looks like: **D < 0.05**, which means the
> maximum gap between empirical and theoretical CDFs is less than 5 percentage points
> anywhere in the distribution.  That is an excellent fit regardless of the p-value.

**`scipy` note:** We use `stats.ks_1samp()` with the fitted CDF.  Because we estimated
the parameters from the same data, the true null distribution of the KS statistic is
slightly different from the theoretical one — so the p-value is optimistic.  For a
rigorous test you would use a bootstrap; here D and the visual check are the primary
evidence.

In [ ]:
print('=' * 66)
print('KS TEST RESULTS  --  GAMMA FIT ON monthly_spending')
print('=' * 66)
print(f"{'Segment':<14} {'n':>6} {'D statistic':>13} {'p-value':>12} {'Result':>8}")
print('-' * 57)

for seg in SEGMENT_ORDER:
    data      = df.loc[df['segment'] == seg, 'monthly_spending'].values
    alpha_fit = gamma_params[seg]['alpha']
    beta_fit  = gamma_params[seg]['beta']

    result = stats.ks_1samp(
        data,
        stats.gamma.cdf,
        args=(alpha_fit, 0, beta_fit),   # (a, loc, scale)
    )

    verdict = 'PASS' if result.pvalue > 0.05 else (f'D={result.statistic:.3f}' if result.statistic < 0.05 else 'CHECK')
    print(f'{seg:<14} {len(data):>6,} {result.statistic:>13.6f} {result.pvalue:>12.4f} {verdict:>8}')

print()
print('A good fit: D < 0.05 (max gap between empirical and theoretical CDF < 5%).')
print('p > 0.05 additionally means we cannot reject the fitted distribution formally.')

In [ ]:
print('=' * 66)
print('KS TEST RESULTS  --  WEIBULL FIT ON months_to_churn')
print('=' * 66)
print(f"{'Segment':<14} {'n':>6} {'D statistic':>13} {'p-value':>12} {'Result':>8}")
print('-' * 57)

for seg in SEGMENT_ORDER:
    data      = df.loc[df['segment'] == seg, 'months_to_churn'].values
    shape_fit = weibull_params[seg]['shape']
    scale_fit = weibull_params[seg]['scale']

    result = stats.ks_1samp(
        data,
        stats.weibull_min.cdf,
        args=(shape_fit, 0, scale_fit),  # (c, loc, scale)
    )

    verdict = 'PASS' if result.pvalue > 0.05 else (f'D={result.statistic:.3f}' if result.statistic < 0.05 else 'CHECK')
    print(f'{seg:<14} {len(data):>6,} {result.statistic:>13.6f} {result.pvalue:>12.4f} {verdict:>8}')

print()
print('A good fit: D < 0.05 (max gap between empirical and theoretical CDF < 5%).')
print('p > 0.05 additionally means we cannot reject the fitted distribution formally.')

In [ ]:
# Visual KS check: empirical CDF vs. fitted CDF, with the KS gap (D) highlighted in red.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

configs = [
    ('monthly_spending', 'Gamma',   gamma_params,   stats.gamma,       'Monthly Spending (R)'),
    ('months_to_churn',  'Weibull', weibull_params, stats.weibull_min, 'Months to Churn'),
]

for row_idx, (col, dist_name, params_dict, dist_obj, xlabel) in enumerate(configs):
    for col_idx, seg in enumerate(SEGMENT_ORDER):
        ax     = axes[row_idx][col_idx]
        colour = SEGMENT_COLOURS[seg]

        data     = df.loc[df['segment'] == seg, col].values
        x_sorted = np.sort(data)
        ecdf     = np.arange(1, len(data) + 1) / len(data)

        # Fitted CDF at the same x-points
        if dist_name == 'Gamma':
            p = params_dict[seg]
            fitted_cdf = dist_obj.cdf(x_sorted, a=p['alpha'], loc=0, scale=p['beta'])
        else:
            p = params_dict[seg]
            fitted_cdf = dist_obj.cdf(x_sorted, c=p['shape'], loc=0, scale=p['scale'])

        ax.plot(x_sorted, ecdf,       color=colour, linewidth=1.5, alpha=0.8, label='Empirical CDF')
        ax.plot(x_sorted, fitted_cdf, color='black', linewidth=1.5, linestyle='--',
                label=f'Fitted {dist_name} CDF')

        # Highlight the maximum KS gap (D)
        gaps    = np.abs(ecdf - fitted_cdf)
        gap_idx = np.argmax(gaps)
        ax.vlines(x_sorted[gap_idx],
                  min(ecdf[gap_idx], fitted_cdf[gap_idx]),
                  max(ecdf[gap_idx], fitted_cdf[gap_idx]),
                  color='red', linewidth=2, label=f'D = {gaps.max():.4f}')

        ax.set_title(f'{seg.replace("_", " ").title()} \u2014 {dist_name}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('Cumulative Probability', fontsize=9)
        ax.legend(fontsize=7)

fig.suptitle('Empirical CDF vs. Fitted CDF \u2014 Red bar = KS gap (D), should be negligible',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ks_cdf_comparison.png', bbox_inches='tight')
plt.show()

---
## 5 — Save Fitted Parameters to JSON

We collect all fitted parameters into a single dictionary and write it to
`reports/fitted_params.json`.  This file is the contract between this notebook
and the Monte Carlo simulation notebook:

- **Why JSON?** Human-readable, easy to version-control, trivially loaded by Python
  with `json.load()`.  No dependency on a specific library's pickle format.
- **What the simulation will do:** For each simulated customer in a given segment,
  it will draw `monthly_spending ~ Gamma(alpha, beta)` and
  `lifetime_months ~ Weibull(shape, scale)`, then discount the resulting revenue
  stream back to present value.  Doing this 100 000 times produces a full CLV
  distribution with confidence intervals.

The fitted parameters replace the hard-coded true values from the data generator —
in a real project, you would never have access to the true parameters, so this step
is exactly what you would do with production customer data.

In [ ]:
fitted_params = {
    'gamma':   gamma_params,
    'weibull': weibull_params,
}

OUTPUT_PATH = REPORTS_DIR / 'fitted_params.json'

with open(OUTPUT_PATH, 'w') as f:
    json.dump(fitted_params, f, indent=2)

print(f'Saved fitted parameters to {OUTPUT_PATH}')
print()
print('Contents:')
print(json.dumps(fitted_params, indent=2))

In [ ]:
# Final comparison: fitted vs. true parameters
print('=' * 75)
print('PARAMETER RECOVERY SUMMARY')
print('How close did MLE get to the true parameters used to generate the data?')
print('=' * 75)

print('\nGAMMA (monthly_spending)')
print(f"{'Segment':<14} {'True alpha':>10} {'Fit alpha':>10} {'delta%':>8}   {'True beta':>10} {'Fit beta':>9} {'delta%':>8}")
print('-' * 74)
for seg in SEGMENT_ORDER:
    ta, tb = TRUE_PARAMS[seg]['gamma']['alpha'], TRUE_PARAMS[seg]['gamma']['beta']
    fa, fb = gamma_params[seg]['alpha'], gamma_params[seg]['beta']
    print(f'{seg:<14} {ta:>10} {fa:>10.3f} {100*(fa-ta)/ta:>+7.2f}%   {tb:>10} {fb:>9.3f} {100*(fb-tb)/tb:>+7.2f}%')

print('\nWEIBULL (months_to_churn)')
print(f"{'Segment':<14} {'True k':>8} {'Fit k':>8} {'delta%':>8}   {'True lambda':>12} {'Fit lambda':>11} {'delta%':>8}")
print('-' * 74)
for seg in SEGMENT_ORDER:
    tk, ts = TRUE_PARAMS[seg]['weibull']['shape'], TRUE_PARAMS[seg]['weibull']['scale']
    fk, fs = weibull_params[seg]['shape'], weibull_params[seg]['scale']
    print(f'{seg:<14} {tk:>8} {fk:>8.3f} {100*(fk-tk)/tk:>+7.2f}%   {ts:>12} {fs:>11.3f} {100*(fs-ts)/ts:>+7.2f}%')

print('\nSmall percentage errors confirm MLE recovered the generating parameters accurately.')
print('In a real project there are no true params to compare—KS tests and visual checks are all we have.')

---
## Summary

| Step | What we did | Why it matters for the simulation |
|---|---|---|
| Distribution fitting | Used MLE (`scipy.stats.*.fit`) to find the Gamma and Weibull parameters that best explain the observed data | Converts a table of past observations into a generative model we can sample from |
| Parameter interpretation | α controls Gamma spread; β scales values. Weibull k controls hazard direction; λ scales lifetime | Understanding parameters lets you sanity-check results and communicate them to stakeholders |
| KS test | Measured the maximum gap D between empirical and fitted CDFs; checked p-value | Provides quantitative (not just visual) confirmation that the fitted distribution is appropriate |
| Parameter export | Saved to `reports/fitted_params.json` | The simulation notebook loads this file — fitting and simulation are cleanly decoupled |

**Next step:** `03_monte_carlo_clv.ipynb` — load `fitted_params.json`, draw thousands
of simulated customers per segment, compute discounted lifetime value for each, and
build CLV distributions with confidence intervals.